# Qwen3.8-27B bounded v3 teacher study on Google Colab

This is the successor to the bounded v2 high-reasoning study. v2 achieved
100% normal stops and clean contamination, but only 76.5% of normally stopped
records fit the Qwen3.5-4B 2,048-token training envelope. The measured failure
was final-answer verbosity rather than prompt length.

v3 keeps the pinned Qwen3.8-27B teacher, native BF16 weights, vLLM 0.28.0,
high reasoning effort, 16,384-token context, 8,192-token generation cap,
sampling parameters, seed, and 16-record shard size. The only scientific
intervention is a SHA-bound per-record final-answer budget computed with the
canonical Qwen3.5-4B tokenizer: min(1792, 2048 - prompt_tokens - 128).

The first v3 generation is the same deterministic 200-record source subset.
Do not scale to 2,000 unless normal stops are >=90%, contamination is clean,
and student-length acceptance is >=85% for this v3 scale decision.


## 1. Mount Google Drive

Select an **A100 GPU** runtime in Colab first, then run this cell.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Define durable Drive paths

The repository ZIP, input corpus, subsets, checkpoints, and finalized data all live
on Google Drive. `/content` is treated as disposable scratch space.

In [ ]:
import os
from pathlib import Path

os.environ["TQC_DRIVE"] = "/content/drive/MyDrive/tiny-qwen-coder"
os.environ["CODE_DIR"] = f"{os.environ['TQC_DRIVE']}/code"
os.environ["CODE_ARCHIVE"] = f"{os.environ['CODE_DIR']}/tiny-qwen-coder-v3.zip"
os.environ["RUN_ROOT"] = f"{os.environ['TQC_DRIVE']}/distillation/qwen38-27b-v1"
os.environ["INPUT_DIR"] = f"{os.environ['RUN_ROOT']}/input"
os.environ["SUBSET_DIR"] = f"{os.environ['RUN_ROOT']}/subsets"
os.environ["SMOKE_DIR"] = f"{os.environ['TQC_DRIVE']}/distillation/qwen38-27b-v1-smoke"
os.environ["PILOT_DIR"] = f"{os.environ['TQC_DRIVE']}/distillation/qwen38-27b-v1-2000"

for key in (
    "CODE_DIR",
    "RUN_ROOT",
    "INPUT_DIR",
    "SUBSET_DIR",
    "SMOKE_DIR",
    "PILOT_DIR",
):
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)

print("Repository ZIP expected at:", os.environ["CODE_ARCHIVE"])

## 3. Put the frozen repository ZIP on Drive

Upload your repository archive to:

`MyDrive/tiny-qwen-coder/code/tiny-qwen-coder-v3.zip`

If the file is not already there, this optional cell lets you select the ZIP from
your computer and copies it to the durable Drive location.

Once a real generation run has started, **do not replace this ZIP with newer code**.
Use a new archive and a new checkpoint directory for a different code version.

In [ ]:
from pathlib import Path

archive = Path(os.environ["CODE_ARCHIVE"])

if archive.exists():
    print("Repository ZIP already exists:", archive)
else:
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Upload exactly one repository ZIP.")

    uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
    if not uploaded_name.lower().endswith(".zip"):
        raise RuntimeError("The uploaded file must be a .zip archive.")

    archive.write_bytes(uploaded_bytes)
    print("Saved repository ZIP to:", archive)

## 4. Verify and extract the frozen repository ZIP

On first use, this writes `tiny-qwen-coder-v3.zip.sha256` next to the archive on Drive.
On every later Colab allocation, the ZIP must match that checksum exactly.

The cell also finds the repository root automatically whether the archive contains
the files directly or wraps them in a directory such as `tiny-qwen-coder-master/`.

Because a GitHub ZIP intentionally contains no `.git` directory, this step also creates
a **local deterministic Git commit** over the extracted source tree. This is provenance
only: no remote is configured and no GitHub credentials, clone, pull, or push are used.
The commit message binds the sealed repository ZIP SHA-256 so generic dataset manifests
can record a valid source-tree Git identity during finalization.

In [ ]:
import hashlib
import shutil
import subprocess
from pathlib import Path
from zipfile import ZipFile

archive = Path(os.environ["CODE_ARCHIVE"])
if not archive.is_file():
    raise FileNotFoundError(
        f"Repository ZIP not found: {archive}\n"
        "Upload tiny-qwen-coder-v3.zip to the Google Drive code directory first."
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


archive_sha256 = sha256_file(archive)
checksum_path = archive.with_suffix(archive.suffix + ".sha256")

if checksum_path.exists():
    expected_sha256 = checksum_path.read_text(encoding="ascii").split()[0]
    if archive_sha256 != expected_sha256:
        raise RuntimeError(
            "Repository ZIP checksum changed. Do not resume this experiment with different code."
        )
else:
    checksum_path.write_text(
        f"{archive_sha256}  {archive.name}\n",
        encoding="ascii",
    )

scratch_root = Path("/content/tiny-qwen-coder-code")
if scratch_root.exists():
    shutil.rmtree(scratch_root)
scratch_root.mkdir(parents=True)

with ZipFile(archive) as zip_file:
    zip_file.extractall(scratch_root)

repo_candidates = sorted(
    {
        pyproject.parent
        for pyproject in scratch_root.rglob("pyproject.toml")
        if (pyproject.parent / "scripts/teacher_distillation/README.md").is_file()
    }
)
if len(repo_candidates) != 1:
    raise RuntimeError(
        "Expected exactly one tiny-qwen-coder repository in the ZIP; "
        f"found {len(repo_candidates)} candidates."
    )

repo = repo_candidates[0]
os.environ["TQC_REPO"] = str(repo)
os.environ["TQC_CODE_ARCHIVE_SHA256"] = archive_sha256

# GitHub source ZIPs deliberately omit .git. Create a deterministic local commit
# so the generic dataset-manifest provenance code can still record an exact
# source-tree identity without any remote, credentials, clone, pull, or push.
git_env = os.environ.copy()
git_env.update(
    {
        "GIT_AUTHOR_NAME": "Tiny Qwen Coder Archive",
        "GIT_AUTHOR_EMAIL": "archive@tiny-qwen-coder.invalid",
        "GIT_COMMITTER_NAME": "Tiny Qwen Coder Archive",
        "GIT_COMMITTER_EMAIL": "archive@tiny-qwen-coder.invalid",
        "GIT_AUTHOR_DATE": "2000-01-01T00:00:00+00:00",
        "GIT_COMMITTER_DATE": "2000-01-01T00:00:00+00:00",
    }
)
subprocess.run(["git", "init", "--quiet"], cwd=repo, check=True, env=git_env)
subprocess.run(
    ["git", "-c", "core.autocrlf=false", "add", "--all"],
    cwd=repo,
    check=True,
    env=git_env,
)
subprocess.run(
    [
        "git",
        "commit",
        "--quiet",
        "--no-gpg-sign",
        "-m",
        f"Frozen repository archive sha256:{archive_sha256}",
    ],
    cwd=repo,
    check=True,
    env=git_env,
)
archive_git_sha = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=repo, text=True, env=git_env
).strip()
archive_git_status = subprocess.check_output(
    ["git", "status", "--porcelain"], cwd=repo, text=True, env=git_env
)
if archive_git_status.strip():
    raise RuntimeError("Extracted repository is unexpectedly dirty after provenance commit.")
os.environ["TQC_ARCHIVE_GIT_SHA"] = archive_git_sha

print("repository:", repo)
print("archive SHA-256:", archive_sha256)
print("archive provenance Git SHA:", archive_git_sha)
print("checksum:", checksum_path)

## 5. Create an isolated uv environment and install the vLLM CUDA 13.0 runtime

Do **not** use Colab's system Python for teacher generation. Colab can preinstall
PyTorch and TorchAudio built against different CUDA versions.

This step installs/updates `uv`, creates `/content/tqc-teacher-venv` with `uv venv`,
and installs vLLM 0.28.0 plus this repository in one resolution using
`--torch-backend=cu130`.

The important point is that **we do not manually pin `+cu130` PyTorch wheels**.
vLLM owns its compatible PyTorch constraint, while uv chooses the CUDA 13.0 wheel
index. This follows the vLLM-supported uv installation path.

Using `uv venv` also avoids depending on Colab's `python3-venv`/`ensurepip` OS packages.
The NVIDIA driver may still report CUDA 13.0 in `nvidia-smi`; that is expected.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.chdir(os.environ["TQC_REPO"])
print("working directory:", os.getcwd())

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "uv"],
    check=True,
)

uv_executable = shutil.which("uv")
if uv_executable is None:
    raise RuntimeError("uv was installed but its executable is not on PATH.")

venv = Path("/content/tqc-teacher-venv")
if venv.exists():
    shutil.rmtree(venv)

subprocess.run(
    [uv_executable, "venv", str(venv), "--python", sys.executable],
    check=True,
)

os.environ["TQC_UV"] = uv_executable
os.environ["TQC_VENV"] = str(venv)
os.environ["TQC_PYTHON"] = str(venv / "bin" / "python")
os.environ["PATH"] = f"{venv / 'bin'}{os.pathsep}{os.environ['PATH']}"

print("uv:", uv_executable)
print("teacher python:", os.environ["TQC_PYTHON"])

In [ ]:
def run_uv(args: list[str]) -> None:
    command = [os.environ["TQC_UV"], *args]
    print("$", " ".join(command), flush=True)
    result = subprocess.run(
        command,
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(result.stdout, end="", flush=True)
    result.check_returncode()


run_uv(
    [
        "pip",
        "install",
        "--python",
        os.environ["TQC_PYTHON"],
        "--torch-backend=cu130",
        "-r",
        "requirements/colab-teacher.txt",
        "-e",
        ".",
    ]
)

## 6. Verify the isolated CUDA/vLLM runtime and the A100

This verification imports only the packages required by the teacher runtime: PyTorch
and vLLM. TorchAudio and TorchVision are deliberately **not** required or imported.

The cell captures the subprocess output so an import/runtime failure prints the real
traceback instead of only a `CalledProcessError`.

`nvidia-smi` may report **CUDA Version 13.0** because that is the maximum CUDA version
supported by the driver. The important value below is `torch CUDA: 13.0`.

In [ ]:
import subprocess
import textwrap

!nvidia-smi

verification_code = textwrap.dedent(
    """
    import shutil
    import subprocess
    import sys

    import torch
    import vllm

    print("python:", sys.executable, flush=True)
    print("torch:", torch.__version__, flush=True)
    print("torch CUDA:", torch.version.cuda, flush=True)
    print("vllm:", vllm.__version__, flush=True)
    ninja = shutil.which("ninja")
    print("ninja:", ninja or "MISSING", flush=True)
    if ninja is None:
        raise RuntimeError("Ninja is not available on PATH inside the teacher runtime.")
    print("ninja version:", subprocess.check_output([ninja, "--version"], text=True).strip(), flush=True)
    print(
        "gpu:",
        torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE",
        flush=True,
    )

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU is not available inside the teacher environment.")
    if torch.version.cuda != "13.0":
        raise RuntimeError(f"Expected PyTorch CUDA 13.0, found {torch.version.cuda!r}.")
    if vllm.__version__ != "0.28.0":
        raise RuntimeError(f"Unexpected vLLM version: {vllm.__version__}")
    """
)

result = subprocess.run(
    [os.environ["TQC_PYTHON"], "-c", verification_code],
    check=False,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(result.stdout, end="", flush=True)
result.check_returncode()

## 7. Define preserved v1 evidence and fresh v3 study paths

The v1 input/subset/checkpoint remain read-only evidence. v3 uses a
new input, checkpoint, diagnostics, finalization, and qualification
namespace.


In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

os.environ["V1_PILOT_DIR"] = os.environ["PILOT_DIR"]
os.environ["V1_DIAG_DIR"] = f"{os.environ['V1_PILOT_DIR']}/diagnostics-v3-analysis"
os.environ["V3_DIR"] = f"{os.environ['TQC_DRIVE']}/distillation/qwen38-27b-v3-200"
os.environ["V3_INPUT_DIR"] = f"{os.environ['V3_DIR']}/input"
os.environ["V3_SOURCE_INPUT"] = f"{os.environ['V3_INPUT_DIR']}/p0-200.jsonl"
os.environ["V3_INPUT"] = f"{os.environ['V3_INPUT_DIR']}/p0-200-v3.jsonl"
os.environ["V3_CHECKPOINT_DIR"] = f"{os.environ['V3_DIR']}/checkpoint"
os.environ["V3_DIAG_DIR"] = f"{os.environ['V3_DIR']}/diagnostics"
os.environ["V3_FINAL_DIR"] = f"{os.environ['V3_DIR']}/final"
os.environ["V3_QUALIFICATION"] = f"{os.environ['V3_DIR']}/qualification.json"

for key in (
    "V1_DIAG_DIR",
    "V3_DIR",
    "V3_INPUT_DIR",
    "V3_CHECKPOINT_DIR",
    "V3_DIAG_DIR",
    "V3_FINAL_DIR",
):
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)


def run_teacher(*args: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    command = [os.environ["TQC_PYTHON"], *args]
    print("$", shlex.join(command), flush=True)
    return subprocess.run(command, check=check, text=True)


print("preserved v1 pilot:", os.environ["V1_PILOT_DIR"])
print("fresh v3 study:", os.environ["V3_DIR"])

## 8. Optionally diagnose the preserved v1 2,000-candidate run

This diagnostic is historical evidence only; it is **not a dependency of the v3 study**.
If the old v1 checkpoint metadata is not present on Drive, the next cell prints a warning
and skips the diagnostic. Continue to Step 9.


In [ ]:
required_v1 = (
    Path(os.environ["INPUT_DIR"]) / "accepted.jsonl",
    Path(os.environ["SUBSET_DIR"]) / "p0-2000.jsonl",
    Path(os.environ["V1_PILOT_DIR"]) / "checkpoint" / "run-identity.json",
)
missing = [str(path) for path in required_v1 if not path.exists()]
if missing:
    print("Skipping optional preserved-v1 diagnostic; missing: " + ", ".join(missing))
    print("Continue to Step 9; v3 generation does not depend on this checkpoint.")
else:
    run_teacher(
        "scripts/teacher_distillation/diagnose_teacher_data.py",
        "--distillation-config",
        "configs/distillation/python/qwen38_27b_v1.yaml",
        "--input",
        str(Path(os.environ["SUBSET_DIR"]) / "p0-2000.jsonl"),
        "--checkpoint-dir",
        str(Path(os.environ["V1_PILOT_DIR"]) / "checkpoint"),
        "--output-dir",
        os.environ["V1_DIAG_DIR"],
    )

## 9. Build and seal the deterministic 200-record v3 input

The already sealed v1 accepted input is deterministically sampled, then
the teacher-only concise-answer policy is applied and SHA-256 sealed.
Existing sealed v3 input is verified and reused.


In [ ]:
def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


v3_input = Path(os.environ["V3_INPUT"])
v3_sidecar = v3_input.with_suffix(v3_input.suffix + ".sha256")
if v3_input.exists() or v3_sidecar.exists():
    if not v3_input.is_file() or not v3_sidecar.is_file():
        raise RuntimeError("v3 input payload/sidecar pair is incomplete; do not guess at recovery")
    expected = v3_sidecar.read_text(encoding="ascii").split()[0]
    actual = file_sha256(v3_input)
    if actual != expected:
        raise RuntimeError("existing v3 input checksum mismatch")
    print("Reusing sealed v3 input:", v3_input)
    print("SHA-256:", actual)
else:
    run_teacher(
        "scripts/teacher_distillation/select_teacher_input.py",
        "--input",
        str(Path(os.environ["INPUT_DIR"]) / "accepted.jsonl"),
        "--output",
        os.environ["V3_SOURCE_INPUT"],
        "--count",
        "200",
    )
    run_teacher(
        "scripts/teacher_distillation/prepare_teacher_v3_input.py",
        "--input",
        os.environ["V3_SOURCE_INPUT"],
        "--output",
        os.environ["V3_INPUT"],
    )

## 10. Preflight the v3 generation identity without loading the teacher

This validates the v3 config, sealed input, checkpoint namespace, and run
identity before expensive inference.


In [ ]:
run_teacher(
    "scripts/teacher_distillation/generate_teacher_data.py",
    "--config",
    "configs/distillation/python/qwen38_27b_v3.yaml",
    "--input",
    os.environ["V3_INPUT"],
    "--checkpoint-dir",
    os.environ["V3_CHECKPOINT_DIR"],
    "--work-dir",
    "/content/tqc-distillation-v3-200",
    "--status-only",
)

## 11. Generate or resume the bounded 200-record v3 study

This is the only cell here that loads Qwen3.8-27B. Re-running verifies
and skips already sealed shards in the fresh v3 checkpoint namespace.


In [ ]:
run_teacher(
    "scripts/teacher_distillation/generate_teacher_data.py",
    "--config",
    "configs/distillation/python/qwen38_27b_v3.yaml",
    "--input",
    os.environ["V3_INPUT"],
    "--checkpoint-dir",
    os.environ["V3_CHECKPOINT_DIR"],
    "--work-dir",
    "/content/tqc-distillation-v3-200",
)

## 12. Verify the durable v3 checkpoint without loading the teacher


In [ ]:
run_teacher(
    "scripts/teacher_distillation/generate_teacher_data.py",
    "--config",
    "configs/distillation/python/qwen38_27b_v3.yaml",
    "--input",
    os.environ["V3_INPUT"],
    "--checkpoint-dir",
    os.environ["V3_CHECKPOINT_DIR"],
    "--work-dir",
    "/content/tqc-distillation-v3-200",
    "--status-only",
)

## 13. Diagnose v3 against the actual student conversation

The teacher-only concise policy is stripped before student token counts
are measured. Diagnostics retain reasoning character counts, never hidden
reasoning text.


In [ ]:
run_teacher(
    "scripts/teacher_distillation/diagnose_teacher_data.py",
    "--distillation-config",
    "configs/distillation/python/qwen38_27b_v3.yaml",
    "--input",
    os.environ["V3_INPUT"],
    "--checkpoint-dir",
    os.environ["V3_CHECKPOINT_DIR"],
    "--output-dir",
    os.environ["V3_DIAG_DIR"],
)

## 14. Finalize v3 with fail-closed protected-benchmark checks

Finalization checks HumanEval, MBPP, and the repository holdout after
generation and refuses to write training files unless contamination is
clean. The teacher-only v3 instruction is stripped before corpus writing.


In [ ]:
run_teacher(
    "scripts/teacher_distillation/finalize_teacher_data.py",
    "--distillation-config",
    "configs/distillation/python/qwen38_27b_v3.yaml",
    "--data-config",
    "configs/data/python/qwen38_27b_distilled_v3.yaml",
    "--input",
    os.environ["V3_INPUT"],
    "--checkpoint-dir",
    os.environ["V3_CHECKPOINT_DIR"],
    "--output-dir",
    os.environ["V3_FINAL_DIR"],
)

## 15. Mechanically qualify the bounded study

Qualification requires >=90% normal stops, >=85% student-length
acceptance among normal stops, and contamination status `clean`.
A failing command still writes qualification.json for inspection.


In [ ]:
qualification = run_teacher(
    "scripts/teacher_distillation/qualify_teacher_study.py",
    "--diagnostics-summary",
    str(Path(os.environ["V3_DIAG_DIR"]) / "teacher-length-summary.json"),
    "--dataset-manifest",
    str(Path(os.environ["V3_FINAL_DIR"]) / "dataset-manifest.json"),
    "--minimum-student-length-accept-rate-given-stop",
    "0.85",
    "--output",
    os.environ["V3_QUALIFICATION"],
    check=False,
)
print("qualification exit code:", qualification.returncode)

## 16. Inspect the scaling decision


In [ ]:
import json

qualification_path = Path(os.environ["V3_QUALIFICATION"])
if not qualification_path.is_file():
    raise FileNotFoundError("qualification.json was not written")
decision = json.loads(qualification_path.read_text(encoding="utf-8"))
print(json.dumps(decision, indent=2, sort_keys=True))
if decision.get("qualified") is not True:
    print("DO NOT SCALE. Revise the policy and rerun a bounded study.")
else:
    print("QUALIFIED. A fresh 2,000-record v3 experiment may be prepared.")

## 17. Scaling guard

This notebook never automatically spends GPU time on the 2,000-record
successor. The guard raises unless the bounded study qualified.


In [ ]:
if decision.get("qualified") is not True:
    raise RuntimeError("v3-200 did not qualify; 2,000-record generation is blocked")
print("v3-200 qualified; preserve this evidence before scaling.")